# FM Agent — Workshop Tutorial

The agent built in this notebook is a **geospatial event analysis assistant** powered by Prithvi-EO-2.0. Given a natural-language request, it resolves the location (geocoding), checks Harmonized Landsat–Sentinel-2 imagery availability for the requested date range, and runs Prithvi inference for one of three supported tasks — **flood detection**, **burn-scar mapping**, or **crop-type classification** — then returns a narrative summary with output links plus a behind-the-scenes JSON audit log.

Everything that defines its behavior (scope, contexts, tool inventory, output format, guardrails, reasoning) lives as plain markdown files inside an **artifact** directory; the notebook wires that artifact to a model using `pydantic-ai` and `pydantic-ai-backends` (read-only `LocalBackend` + `ConsoleCapability`), so the agent reads its own prompt material from disk on demand via tool calls.

## 1. Imports and global config

In [ ]:
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load API keys and PRITHVI_SERVER_URL from .env
load_dotenv()



ARTIFACT_PATH = Path("artifacts/Prithvi_WOrkshop_Agent_artifact")

# ── Backend & model selection ─────────────────────────────────────────────────
# Switch backends by setting AGENT_BACKEND in your .env or environment:
#   AGENT_BACKEND=openai   → uses OpenAI API  (default; good for local dev)
#   AGENT_BACKEND=bedrock  → uses AWS Bedrock (good for SageMaker)
#
# On SageMaker the execution IAM role is picked up automatically by boto3 —
# no explicit AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY needed.
#
# Override the model with AGENT_MODEL, e.g.:
#   AGENT_MODEL=bedrock:us.anthropic.claude-3-5-sonnet-20241022-v2:0
#
# See GeoAIAgentConfig docstring for the full model compatibility table.

BACKEND    = os.getenv("AGENT_BACKEND", "bedrock")       # "openai" | "bedrock"
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

_DEFAULT_MODELS = {
    "openai":  "openai:gpt-5.2",
    "bedrock": "bedrock:us.anthropic.claude-sonnet-4-6",
}
MODEL_NAME = os.getenv("AGENT_MODEL") or _DEFAULT_MODELS[BACKEND]

print(f"Backend : {BACKEND}")
print(f"Model   : {MODEL_NAME}")
if BACKEND == "bedrock":
    print(f"Region  : {AWS_REGION}")

Backend : bedrock
Model   : bedrock:us.anthropic.claude-sonnet-4-6
Region  : us-east-1


## 2. Artifact structure

<details>
<summary>Details</summary>

An agent artifact follows this convention:

```
<workspace>/
  ├── resources/        ← user-uploaded reference material
  │   └── <files>
  ├── scope.md          ← agent purpose, users, workflow, success
  ├── contexts/         ← existing systems + context workspace
  │   ├── index.md      ← manifest written when contexts confirmed
  │   └── <topic>.md
  ├── tools/            ← tool inventory
  │   ├── index.md      ← manifest of tools
  │   └── <tool>/index.md
  ├── output.md         ← output format
  ├── reasoning.md      ← reasoning strategy
  ├── guardrails/       ← safety & guardrails
  │   ├── index.md
  │   └── <rule>.md
  └── agents.md         ← final assembled agent prompt (the entry point)
```

**`agents.md` is the entry point** — we load it verbatim as the system prompt. It internally references the other files by relative path (e.g. `contexts/hls_conventions.md`), and the agent pulls them in on demand through its read-only console tools. The notebook itself does **not** stitch files together.

</details>

## 3. Inspect the artifact (no agent attached yet)

<details>
<summary>Details</summary>

Two small helpers let us look at the artifact before we attach an agent:

- `get_artifact_tree(path)` → stringified directory tree (also reused to inject into the system prompt later).
- `inspect_artifact(path)` → prints it.

</details>

In [2]:
def get_artifact_tree(artifact_path: str | Path) -> str:
    """Return a stringified directory tree of the artifact, relative to its root.

    The root directory name itself is intentionally omitted so paths in the tree
    are exactly what the agent should pass to its read tools — the backend is
    already rooted at the artifact path and cannot reach anything above it.
    """
    root = Path(artifact_path).resolve()
    lines: list[str] = []

    def walk(d: Path, prefix: str = "") -> None:
        entries = sorted(d.iterdir(), key=lambda p: (p.is_file(), p.name))
        for i, p in enumerate(entries):
            connector = "└── " if i == len(entries) - 1 else "├── "
            lines.append(f"{prefix}{connector}{p.name}{'/' if p.is_dir() else ''}")
            if p.is_dir():
                extension = "    " if i == len(entries) - 1 else "│   "
                walk(p, prefix + extension)

    walk(root)
    return "\n".join(lines)


def inspect_artifact(artifact_path: str | Path) -> None:
    """Print the artifact tree for human inspection before attaching an agent."""
    root = Path(artifact_path).resolve()
    print(f"{root.name}/  (mounted as read-only root)")
    print(get_artifact_tree(artifact_path))

In [3]:
inspect_artifact(ARTIFACT_PATH)

Prithvi_WOrkshop_Agent_artifact/  (mounted as read-only root)
├── contexts/
│   ├── compute_and_storage.md
│   ├── earthdata_access.md
│   ├── event_catalogs.md
│   ├── hls_catalog_access.md
│   ├── hls_conventions.md
│   ├── index.md
│   ├── prithvi_inference_mcp.md
│   ├── prithvi_task_capabilities.md
│   └── user_confirmations.md
├── guardrails/
│   ├── index.md
│   └── safety_and_guardrails.md
├── tools/
│   ├── datasets/
│   │   ├── index.md
│   │   ├── query_active_fires.md
│   │   ├── query_crop_landcover.md
│   │   ├── query_fire_history.md
│   │   └── query_surface_water.md
│   ├── feasibility/
│   │   └── index.md
│   ├── geocode/
│   │   ├── geocode_location.md
│   │   └── index.md
│   ├── hls/
│   │   ├── check_hls_availability.md
│   │   └── index.md
│   ├── prithvi/
│   │   ├── get_prithvi_job_status.md
│   │   ├── get_prithvi_results.md
│   │   ├── index.md
│   │   └── run_prithvi_inference.md
│   └── index.md
├── agents.md
├── output.md
├── reasoning.md
└── scope.md


## 4. Assembling the agent from the artifact

<details>
<summary>Details</summary>

`GeoAIAgent` extends akd-ext's `PydanticAIBaseAgent` and handles all wiring internally:

- Reads `agents.md` + artifact tree → builds the system prompt.
- Creates a `LocalBackend` (read-only, rooted at the artifact) and wraps it in a
  `ConsoleCapability` so the model can call `ls`, `read_file`, `glob`, and `grep`
  on the artifact at runtime.
- Registers all 9 tools from `tools/` (each a `BaseTool` subclass decorated with
  `@mcp_tool`) via akd-ext's `_adapt_tools` → pydantic-ai `Tool` conversion.
- Exposes `agent.deps` so the raw `agent.run()` / `run_stream_events()` calls used
  in the streaming and Gradio cells below can pass the backend deps explicitly.

`GeoAIAgentConfig` extends `PydanticAIBaseAgentConfig` with `artifact_path`,
`model_name`, and `reasoning_effort` fields. Pass custom values to override defaults.

</details>

In [4]:
from __future__ import annotations

from collections.abc import AsyncIterator
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

from akd._base import InputSchema, StreamEvent, TextOutput
from akd_ext.agents._base.pydantic_ai import (
    PydanticAIBaseAgent,
    PydanticAIBaseAgentConfig,
)
from pydantic import ConfigDict, Field
from pydantic_ai import UsageLimits
from pydantic_ai_backends import READONLY_RULESET, ConsoleCapability, LocalBackend

from tools import TOOLS

_NAVIGATION_DIRECTIVE = """
# Artifact navigation

You are running against a local artifact workspace, mounted read-only at the root
of your console tools (`ls`, `read_file`, `glob`, `grep`). The full layout is:

```
{tree}
```

Navigation rules:
- **Answer from what you already have.** The full directory tree is visible above.
  For questions about what files or topics exist, answer from the tree — do not
  make tool calls to discover something already shown here.
- **For content questions**, read the relevant subdirectory's `index.md` first.
  Each index summarises every file in its directory. Read individual files only
  when the index alone is not sufficient to answer.
- **Never read a file you have already read** in this conversation.
- **Stop as soon as you can answer.** Do not keep reading to be more thorough;
  synthesise from what you have and respond.
- Do not `grep` or `ls` directories already shown in the tree above.
- The filesystem is read-only. Do not write, edit, or execute.
""".strip()


@dataclass
class Deps:
    backend: LocalBackend


class GeoAIAgentInput(InputSchema):
    """Input for the GeoAI geospatial analysis agent."""
    query: str = Field(..., description="Natural language geospatial analysis request")


class GeoAIAgentConfig(PydanticAIBaseAgentConfig):
    """Configuration for the GeoAI workshop agent.

    Backend selection
    -----------------
    Controlled by the global BACKEND / MODEL_NAME variables set in cell 1,
    which read from AGENT_BACKEND / AGENT_MODEL env vars (or .env).

    OpenAI (local dev)
      model_name = "openai:gpt-5.2"
      aws_region = None

    Bedrock (SageMaker)
      model_name = "bedrock:us.anthropic.claude-sonnet-4-20250514-v1:0"
      aws_region = "us-east-1"  (or AWS_DEFAULT_REGION)
      On SageMaker the execution IAM role is used automatically — no credentials needed.

    Model compatibility
    -------------------
    Requirements: tool/function calling + streaming.

    OpenAI models
      openai:gpt-5.2               tool calls  thinking    (default)
      openai:o3                    tool calls  thinking
      openai:gpt-4o                tool calls  no thinking  -> reasoning_effort=None
      openai:gpt-4o-mini           tool calls  no thinking  -> reasoning_effort=None

    Bedrock -- Anthropic Claude 4.x (current generation, recommended on SageMaker)
      bedrock:us.anthropic.claude-sonnet-4-6                   thinking    (default)
      bedrock:eu.anthropic.claude-sonnet-4-6                   thinking    (EU region)
      bedrock:us.anthropic.claude-sonnet-4-5-20250929-v1:0     thinking
      bedrock:us.anthropic.claude-haiku-4-5-20251001-v1:0      thinking

    Bedrock -- Amazon Nova (tool calls, no thinking -> reasoning_effort=None)
      bedrock:us.amazon.nova-pro-v1:0
      bedrock:us.amazon.nova-lite-v1:0

    End-of-life on Bedrock (will 404)
      bedrock:us.anthropic.claude-3-5-sonnet-20241022-v2:0     EOL May 2026
      bedrock:us.anthropic.claude-sonnet-4-20250514-v1:0       Legacy
      bedrock:us.anthropic.claude-3-7-sonnet-20250219-v1:0     EOL

    Not compatible
      bedrock:meta.llama3-*   unreliable parallel tool calling
      bedrock:amazon.titan-*  no tool calling
    """
    model_config = ConfigDict(extra="allow", arbitrary_types_allowed=True)

    artifact_path: Path = Field(
        default=Path("artifacts/Prithvi_WOrkshop_Agent_artifact"),
        description="Path to the CARE artifact directory",
    )
    model_name: str = Field(default=MODEL_NAME)  # override with AGENT_MODEL env var
    aws_region: str | None = Field(
        default=AWS_REGION if BACKEND == "bedrock" else None,
        description=(
            "AWS region for Bedrock. When set, a BedrockConverseModel is built "
            "with the SageMaker execution role (no explicit credentials needed). "
            "Leave None to use the model_name string directly (OpenAI path)."
        ),
    )
    reasoning_effort: Literal["low", "medium", "high"] | None = Field(
        default="medium",
        description=(
            "Extended thinking effort. Only meaningful for models that support it "
            "-- see the compatibility table above. Set to None for non-thinking models."
        ),
    )
    request_limit: int = Field(default=15, description="Max LLM requests per run; caps runaway loops.")


class GeoAIAgent(PydanticAIBaseAgent[GeoAIAgentInput, TextOutput]):
    """Geospatial event analysis agent powered by Prithvi-EO.

    Resolves locations, checks HLS imagery availability, runs Prithvi inference
    for flood/burn/crop tasks, and synthesises results into a narrative summary.
    """

    input_schema = GeoAIAgentInput
    output_schema = TextOutput
    config_schema = GeoAIAgentConfig

    def __init__(self, config: GeoAIAgentConfig | None = None) -> None:
        cfg = config or GeoAIAgentConfig()
        artifact = cfg.artifact_path.resolve()

        backend = LocalBackend(
            root_dir=artifact,
            allowed_directories=[str(artifact)],
            enable_execute=False,
            permissions=READONLY_RULESET,
        )
        self._deps = Deps(backend=backend)
        capability = ConsoleCapability(
            include_execute=False,
            permissions=READONLY_RULESET,
        )

        system_prompt = (
            (artifact / "agents.md").read_text()
            + "\n\n"
            + _NAVIGATION_DIRECTIVE.format(tree=get_artifact_tree(artifact))
        )

        full_cfg = GeoAIAgentConfig(
            artifact_path=cfg.artifact_path,
            model_name=cfg.model_name,
            aws_region=cfg.aws_region,
            reasoning_effort=cfg.reasoning_effort,
            request_limit=cfg.request_limit,
            system_prompt=system_prompt,
            capabilities=[capability, *cfg.capabilities],
            tools=list(TOOLS),
            deps_type=Deps,
        )
        super().__init__(full_cfg)

        # When aws_region is set we replace the model with a BedrockConverseModel that
        # pins the region and uses the SageMaker execution IAM role via boto3's
        # credential chain.  We do this AFTER super().__init__() to avoid passing a
        # non-standard kwarg through the config (which would leak into extra_kwargs and
        # be rejected by pydantic-ai's Agent.__init__).
        if cfg.aws_region:
            from pydantic_ai.models.bedrock import BedrockConverseModel
            from pydantic_ai.providers.bedrock import BedrockProvider
            # Strip the "bedrock:" prefix -- BedrockConverseModel takes the raw model ID
            raw_id = cfg.model_name.removeprefix("bedrock:")
            self._model = BedrockConverseModel(
                raw_id,
                provider=BedrockProvider(region_name=cfg.aws_region),
            )

    @property
    def deps(self) -> Deps:
        """Pre-wired Deps for passing to agent.run() / run_stream_events()."""
        return self._deps

    @property
    def usage_limits(self) -> UsageLimits:
        return UsageLimits(request_limit=self.config.request_limit)

    async def arun(
        self,
        params: GeoAIAgentInput,
        run_context: Any = None,
        **kwargs: Any,
    ) -> TextOutput:
        kwargs.setdefault("deps", self._deps)
        return await super().arun(params, run_context, **kwargs)

    async def astream(
        self,
        params: GeoAIAgentInput,
        run_context: Any = None,
        **kwargs: Any,
    ) -> AsyncIterator[StreamEvent]:
        kwargs.setdefault("deps", self._deps)
        async for event in super().astream(params, run_context, **kwargs):
            yield event


# Show the artifact tree (same view injected into the agent's system prompt)
root = ARTIFACT_PATH.resolve()
print(f"{root.name}/  (mounted as read-only root)")
print(get_artifact_tree(root))

## 6. Initialize the agent

In [ ]:
agent = GeoAIAgent(GeoAIAgentConfig(artifact_path=ARTIFACT_PATH))

# Show the assembled system prompt
display(Markdown(
    f"**Finalized system prompt** — {len(agent.config.system_prompt)} chars\n\n"
    "---\n\n"
    f"{agent.config.system_prompt}"
))

## 7. Run (async)

<details>
<summary>Details</summary>

`Agent.run(...)` is async and returns the full result once the agent is done. Jupyter lets us `await` it directly inside a cell.

</details>

In [ ]:
result = await agent.run(
    "List the tools available in this artifact and briefly summarize what each one does.",
    deps=agent.deps,
    usage_limits=agent.usage_limits,
)
print(result.output)

## 8. Run with streaming (full event trace)

<details>
<summary>Details</summary>

`agent.run_stream_events(...)` exposes the agent's full event timeline as it executes: when a new part starts (text, thinking, tool call), incremental deltas for text and thinking, and tool-call results. We dispatch on event/part type and print a compact trace so you can see what the agent is doing in real time — not just the final answer.

</details>

In [ ]:
from pydantic_ai import AgentRunResultEvent, UsageLimits
from pydantic_ai.messages import (
    PartStartEvent,
    PartDeltaEvent,
    FunctionToolCallEvent,
    FunctionToolResultEvent,
    TextPart,
    TextPartDelta,
    ThinkingPart,
    ThinkingPartDelta,
)


async def stream_with_trace(prompt: str) -> None:
    """Run the agent and print a live trace of tool calls, reasoning, and text deltas."""
    async with agent.run_stream_events(
        prompt, deps=agent.deps, usage_limits=agent.usage_limits
    ) as stream:
        async for event in stream:
            if isinstance(event, FunctionToolCallEvent):
                part = event.part
                print(f"\n[tool call] {part.tool_name}({part.args})", flush=True)
            elif isinstance(event, FunctionToolResultEvent):
                preview = str(event.part.content)
                preview = preview if len(preview) <= 200 else preview[:200] + "..."
                print(f"\n[tool result] {preview}", flush=True)
            elif isinstance(event, PartStartEvent):
                part = event.part
                if isinstance(part, ThinkingPart):
                    print("\n[thinking] ", end="", flush=True)
                    if part.content:
                        print(part.content, end="", flush=True)
                elif isinstance(part, TextPart):
                    print("\n[output] ", end="", flush=True)
                    if part.content:
                        print(part.content, end="", flush=True)
            elif isinstance(event, PartDeltaEvent):
                delta = event.delta
                if isinstance(delta, (TextPartDelta, ThinkingPartDelta)) and delta.content_delta:
                    print(delta.content_delta, end="", flush=True)
            elif isinstance(event, AgentRunResultEvent):
                output = event.result.output
                text = output.content if hasattr(output, "content") else str(output)
                print(f"\n[output] {text}", flush=True)
                print("\n[done]", flush=True)


await stream_with_trace(
    "List the tools available in this artifact and briefly summarize what each one does."
)

## 9. Chat interface (Gradio)

<details>
<summary>Details</summary>

`build_chat_interface(agent, deps)` launches an inline Gradio chat UI bound to the agent. Each turn:

- Streams reasoning summary deltas into a collapsible **Reasoning** block.
- Streams tool calls and their results into a collapsible **Trace** block.
- Streams the model's final answer into the chat bubble itself.

Conversation history is preserved across turns: after each turn the full `pydantic-ai` message list is captured from `AgentRunResultEvent.result.all_messages()` and passed back as `message_history` on the next call.

</details>

In [ ]:
def build_chat_interface(
    agent,
    *,
    title: str = "FM Agent chat",
) -> gr.ChatInterface:
    """Launch an inline Gradio chat UI bound to `agent`.

    Conversation history is kept across turns in a closure-local state dict.
    """
    state = {"message_history": []}

    async def chat_fn(message, history):
        trace: list[str] = []
        thinking_parts: list[str] = []
        text_parts: list[str] = []

        def render() -> str:
            chunks: list[str] = []
            if thinking_parts:
                chunks.append(
                    "<details><summary>Reasoning</summary>\n\n"
                    + "".join(thinking_parts)
                    + "\n\n</details>"
                )
            if trace:
                chunks.append(
                    f"<details><summary>Trace ({len(trace)} step"
                    + ("s" if len(trace) != 1 else "")
                    + ")</summary>\n\n"
                    + "\n".join(trace)
                    + "\n\n</details>"
                )
            if text_parts:
                chunks.append("".join(text_parts))
            return "\n\n".join(chunks) if chunks else "_thinking..._"

        async with agent.run_stream_events(
            message,
            deps=agent.deps,
            message_history=state["message_history"],
            usage_limits=agent.usage_limits,
        ) as stream:
            async for event in stream:
                if isinstance(event, FunctionToolCallEvent):
                    args = str(event.part.args)
                    args = args if len(args) <= 100 else args[:100] + "..."
                    trace.append(f"- `{event.part.tool_name}({args})`")
                    yield render()
                elif isinstance(event, FunctionToolResultEvent):
                    preview = str(event.part.content).replace("\n", " ")
                    preview = preview if len(preview) <= 150 else preview[:150] + "..."
                    trace.append(f"  - ↳ {preview}")
                    yield render()
                elif isinstance(event, PartStartEvent):
                    part = event.part
                    if isinstance(part, TextPart) and part.content:
                        text_parts.append(part.content)
                        yield render()
                    elif isinstance(part, ThinkingPart) and part.content:
                        thinking_parts.append(part.content)
                        yield render()
                elif isinstance(event, PartDeltaEvent):
                    d = event.delta
                    if isinstance(d, TextPartDelta) and d.content_delta:
                        text_parts.append(d.content_delta)
                        yield render()
                    elif isinstance(d, ThinkingPartDelta) and d.content_delta:
                        thinking_parts.append(d.content_delta)
                        yield render()
                elif isinstance(event, AgentRunResultEvent):
                    state["message_history"] = event.result.all_messages()
                    output = event.result.output
                    text = output.content if hasattr(output, "content") else str(output)
                    text_parts.append(text)
                    yield render()

    demo = gr.ChatInterface(
        fn=chat_fn,
        title=title,
        examples=[
            "List the tools available in this artifact.",
            "What contexts are defined and what does each cover?",
            "Summarize the safety guardrails.",
        ],
    )
    demo.launch(inline=True)
    return demo


chat = build_chat_interface(agent)

## 10. What's next

<details>
<summary>Details</summary>

- Try out-of-scope queries to confirm the agent refuses per the `guardrails/` rules.
- Swap `ARTIFACT_PATH` to a different artifact directory — the same `create_agent_from_artifact` call gives you a different agent without code changes.
- Add new tools to `AGENT_TOOLS` in cell 4 and implement them under `tools/` following the same pattern.

</details>